# 🛡️ Python Exception Handling & Defensive Coding
### *try/except/else/finally, Custom Exceptions, Context Managers*

---

> **Mental Model First:**
> Think of exception handling as a safety net under a trapeze act.
> The performer (your code) tries to execute perfectly. If they slip (exception),
> the net (except) catches them. Some nets have layers — finally always deploys,
> no matter what. else fires only when the performer sticks the landing.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [try / except / else / finally — Full Anatomy](#1) |
| 2 | [Exception Hierarchy](#2) |
| 3 | [Custom Exceptions](#3) |
| 4 | [Context Managers — with statement](#4) |
| 5 | [Retry Pattern & Defensive Coding](#5) |
| 6 | [Decision Map & Cheat Sheet](#6) |


<a id='1'></a>

## 1. try / except / else / finally — Full Anatomy

---

```
STRUCTURE:
  try:
      [code that might raise]
  except ExceptionType as e:
      [handle specific exception]
  except (TypeError, ValueError) as e:
      [handle multiple types]
  except Exception as e:
      [catch-all — use sparingly]
  else:
      [runs ONLY if no exception was raised in try]
  finally:
      [ALWAYS runs — exception or not]

EXECUTION PATHS:
  No exception:   try → else → finally
  Exception caught:  try(partial) → except → finally
  Exception uncaught: try(partial) → finally → propagates up

else BLOCK — often forgotten:
  Use it for code that should only run on SUCCESS.
  Keeps "normal path" code out of the try block
  (so you don't accidentally catch exceptions from it).

  try:
      result = compute()   # might fail
  except ComputeError:
      handle_error()
  else:
      log(result)          # only runs if compute() succeeded
      save(result)         # exceptions here will NOT be caught by above except

finally BLOCK — always runs:
  File close, lock release, DB connection close.
  Even if you hit return or raise inside try/except.
```


In [ ]:
# Full try/except/else/finally demonstration

def safe_divide(a, b):
    try:
        result = a / b          # might raise ZeroDivisionError
        print(f"  [try] computed {a}/{b} = {result}")
    except ZeroDivisionError as e:
        print(f"  [except] caught: {e}")
        result = None
    else:
        # Only runs if NO exception in try
        print(f"  [else] success — result is {result}")
    finally:
        # ALWAYS runs
        print(f"  [finally] cleanup (always runs)")
    return result

print("Case 1: normal division")
safe_divide(10, 2)

print("Case 2: division by zero")
safe_divide(10, 0)

# ── Multiple exception types ──────────────────────────────────────────────────
def parse_index(data, idx):
    try:
        val = int(data[idx])    # IndexError if idx out of range
        return 100 / val        # ZeroDivisionError if val == 0
    except IndexError:
        print(f"  IndexError: index {idx} out of range for len={len(data)}")
    except ZeroDivisionError:
        print(f"  ZeroDivisionError: data[{idx}]=0")
    except (TypeError, ValueError) as e:
        print(f"  Type/Value error: {e}")

parse_index(["5", "0", "abc"], 0)   # works: 100/5 = 20
parse_index(["5", "0", "abc"], 1)   # ZeroDivisionError
parse_index(["5", "0", "abc"], 5)   # IndexError
parse_index(["5", "0", "abc"], 2)   # ValueError (int("abc"))

# ── finally with return ──────────────────────────────────────────────────────
def always_cleanup():
    try:
        print("  [try] working...")
        return "from try"       # return doesn't skip finally!
    finally:
        print("  [finally] still runs even after return")

result = always_cleanup()
print(f"  returned: {result}")
print("Exception anatomy demo complete.")


<a id='2'></a>

## 2. Exception Hierarchy

---

```
BaseException
├─ SystemExit          ← sys.exit() — DO NOT catch with `except Exception`
├─ KeyboardInterrupt   ← Ctrl+C — DO NOT catch with `except Exception`
├─ GeneratorExit       ← generator.close() — DO NOT catch with `except Exception`
└─ Exception           ← all "normal" exceptions live here
   ├─ ValueError        ← right type, wrong value  (int("abc"))
   ├─ TypeError         ← wrong type               (1 + "a")
   ├─ IndexError        ← list index out of range  (lst[99])
   ├─ KeyError          ← dict key missing         (d["missing"])
   ├─ AttributeError    ← no such attribute        (None.split())
   ├─ NameError         ← undefined variable       (x not defined)
   ├─ ZeroDivisionError ← divide by zero
   ├─ FileNotFoundError ← (subclass of OSError)
   ├─ StopIteration     ← end of iterator — usually internal
   ├─ RuntimeError      ← generic runtime error
   ├─ NotImplementedError ← abstract method not overridden
   └─ ArithmeticError
      └─ ZeroDivisionError, OverflowError, FloatingPointError

CATCH SPECIFICS, NOT GENERICS:
  ❌  except Exception:     # swallows everything — hides bugs
  ❌  except:               # catches even SystemExit / KeyboardInterrupt!
  ✅  except ValueError:
  ✅  except (KeyError, IndexError):
  ✅  except Exception as e: raise  # re-raise after logging
```


In [ ]:
# Exception hierarchy and specific catching

# ── isinstance check shows hierarchy ─────────────────────────────────────────
errors = [
    ValueError("bad value"),
    TypeError("bad type"),
    IndexError("bad index"),
    ZeroDivisionError("div by zero"),
    KeyError("missing key"),
]

print("Exception → Exception? → BaseException?")
for e in errors:
    print(f"  {type(e).__name__:<22} is Exception:{isinstance(e, Exception)}  "
          f"is BaseException:{isinstance(e, BaseException)}")

# ── Catching parent catches children ─────────────────────────────────────────
def risky(x):
    if x == 0: raise ZeroDivisionError("zero")
    if x < 0:  raise ValueError("negative")
    return 10 / x

for val in [5, 0, -1]:
    try:
        print(f"risky({val}) = {risky(val):.2f}")
    except ArithmeticError as e:  # catches ZeroDivisionError (subclass)
        print(f"  ArithmeticError caught: {e}")
    except ValueError as e:
        print(f"  ValueError caught: {e}")

# ── Exception chaining ────────────────────────────────────────────────────────
def load_config(path):
    try:
        data = {}["missing_key"]   # simulates KeyError from bad config
    except KeyError as e:
        # raise ... from e chains the original exception
        raise RuntimeError(f"Config load failed for {path}") from e

try:
    load_config("/etc/app.conf")
except RuntimeError as e:
    print(f"RuntimeError: {e}")
    print(f"Caused by: {e.__cause__}")   # the original KeyError

print("Exception hierarchy demo complete.")


<a id='3'></a>

## 3. Custom Exceptions

---

```
WHY CUSTOM EXCEPTIONS?
  1. Gives callers something specific to catch
  2. Carries domain-relevant data (not just a message)
  3. Makes error logs self-documenting

PATTERN:
  class DomainError(Exception):
      pass

  class ValidationError(DomainError):
      def __init__(self, field, value, message):
          self.field = field
          self.value = value
          super().__init__(f"{field}={value!r}: {message}")

HIERARCHY DESIGN:
  BaseAppError          ← all app errors inherit this
  ├─ ValidationError    ← input invalid
  ├─ NotFoundError      ← resource doesn't exist
  └─ AuthError          ← auth failure

  Callers can catch BaseAppError for broad handling
  or ValidationError for specific handling.

LC USAGE:
  Custom exceptions rarely appear in LC solutions.
  But they're a common interview question:
  "Design a library / class with proper error handling."
```


In [ ]:
# Custom exception hierarchy

class AppError(Exception):
    '''Base for all application errors.'''
    pass

class ValidationError(AppError):
    '''Input field failed validation.'''
    def __init__(self, field, value, reason):
        self.field = field
        self.value = value
        self.reason = reason
        super().__init__(f"Validation failed: {field}={value!r} — {reason}")

class NotFoundError(AppError):
    '''Resource not found.'''
    def __init__(self, resource_type, key):
        self.resource_type = resource_type
        self.key = key
        super().__init__(f"{resource_type} not found: {key!r}")

class RateLimitError(AppError):
    '''Too many requests.'''
    def __init__(self, limit, window_sec):
        self.limit = limit
        self.window_sec = window_sec
        super().__init__(f"Rate limit exceeded: {limit} requests per {window_sec}s")

# ── Using custom exceptions ───────────────────────────────────────────────────
def lookup_user(user_id: int, db: dict) -> dict:
    if not isinstance(user_id, int):
        raise ValidationError("user_id", user_id, "must be int")
    if user_id <= 0:
        raise ValidationError("user_id", user_id, "must be positive")
    if user_id not in db:
        raise NotFoundError("User", user_id)
    return db[user_id]

db = {1: {"name": "Alice"}, 2: {"name": "Bob"}}

for uid in [1, 99, "hello", -5]:
    try:
        user = lookup_user(uid, db)
        print(f"Found user {uid}: {user}")
    except ValidationError as e:
        print(f"Validation: field={e.field}, value={e.value!r}, reason={e.reason}")
    except NotFoundError as e:
        print(f"Not found: {e.resource_type} key={e.key!r}")
    except AppError as e:
        print(f"App error: {e}")

# ── isinstance check on hierarchy ────────────────────────────────────────────
err = ValidationError("email", "not-an-email", "missing @")
print(f"is ValidationError: {isinstance(err, ValidationError)}")
print(f"is AppError:        {isinstance(err, AppError)}")
print(f"is Exception:       {isinstance(err, Exception)}")
print("Custom exceptions demo complete.")


<a id='4'></a>

## 4. Context Managers — with statement

---

```
WHAT IS A CONTEXT MANAGER?
  An object that defines SETUP and TEARDOWN behavior.
  The `with` statement calls __enter__ on entry, __exit__ on exit.
  __exit__ is called even if an exception occurs.

  with resource as r:
      use(r)
  # __exit__ called here — guaranteed

EQUIVALENT TO:
  r = resource.__enter__()
  try:
      use(r)
  except:
      resource.__exit__(*sys.exc_info())
      raise
  else:
      resource.__exit__(None, None, None)

BUILT-IN CONTEXT MANAGERS:
  open(file)          → closes file on exit
  threading.Lock()    → releases lock on exit
  decimal.localcontext() → restores decimal context

__exit__ SIGNATURE:
  def __exit__(self, exc_type, exc_val, exc_tb):
      # exc_type is None if no exception
      # return True to SUPPRESS the exception
      # return False/None to let it propagate
      ...

contextlib.contextmanager — simpler approach with yield:
  @contextlib.contextmanager
  def timer():
      start = time.perf_counter()
      yield        ← code inside `with` block runs here
      elapsed = time.perf_counter() - start
      print(f"Elapsed: {elapsed:.3f}s")
```


In [ ]:
import contextlib, time, threading

# ── Pattern 1: class-based context manager ────────────────────────────────────
class Timer:
    '''Measure elapsed time for a code block.'''
    def __init__(self, label=""):
        self.label = label

    def __enter__(self):
        self._start = time.perf_counter()
        print(f"  [enter] {self.label} started")
        return self               # value bound to `as` variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self._start
        print(f"  [exit]  {self.label} elapsed={self.elapsed*1000:.2f}ms  "
              f"exception={exc_type.__name__ if exc_type else None}")
        return False              # don't suppress exceptions

with Timer("list sort") as t:
    data = list(range(10_000, 0, -1))
    data.sort()
print(f"  elapsed attribute: {t.elapsed*1000:.2f}ms")

# ── Exceptions inside `with` still get __exit__ called ───────────────────────
print()
try:
    with Timer("failing block"):
        raise ValueError("something went wrong")
except ValueError as e:
    print(f"  caught: {e}")

# ── Pattern 2: @contextmanager decorator (simpler) ────────────────────────────
@contextlib.contextmanager
def managed_resource(name):
    print(f"  [setup] acquiring {name}")
    try:
        yield f"resource:{name}"   # value available in `with ... as x`
    finally:
        print(f"  [teardown] releasing {name}")   # always runs

with managed_resource("database") as r:
    print(f"  using {r}")

# ── Suppress specific exceptions ─────────────────────────────────────────────
with contextlib.suppress(FileNotFoundError):
    open("does_not_exist.txt")   # would raise FileNotFoundError — suppressed
print("  suppress: no error raised from missing file")

# ── Multiple context managers in one line ─────────────────────────────────────
@contextlib.contextmanager
def fake_open(name):
    yield f"FILE:{name}"

with fake_open("a.txt") as fa, fake_open("b.txt") as fb:
    print(f"  both open: {fa}, {fb}")

print("Context manager demo complete.")


<a id='5'></a>

## 5. Retry Pattern & Defensive Coding

---

```
RETRY PATTERN:
  For transient failures (network, database) — try again N times.
  Key: don't retry on non-transient errors (bad input, auth).

  def with_retry(fn, retries=3, exceptions=(Exception,)):
      for attempt in range(retries):
          try:
              return fn()
          except exceptions as e:
              if attempt == retries - 1:
                  raise   # last attempt — give up
              time.sleep(2 ** attempt)   # exponential backoff

DEFENSIVE CODING PRINCIPLES:
  1. Validate inputs at system boundaries (not deep inside)
  2. Fail fast — raise early with clear message
  3. Never swallow exceptions silently
  4. Use logging.exception() to preserve traceback
  5. Prefer EAFP over LBYL for Python idioms

EAFP (Easier to Ask Forgiveness than Permission):
  ✅ Pythonic
  try:
      val = d["key"]
  except KeyError:
      val = default

LBYL (Look Before You Leap):
  ❌ Un-Pythonic (but OK for expensive operations)
  if "key" in d:
      val = d["key"]
  else:
      val = default

  EAFP is preferred in Python because:
  - Avoids race conditions (check then use)
  - Often faster (single lookup vs two)
  - More natural with Python's exception model
```


In [ ]:
import time, random

# ── Retry decorator with exponential backoff ──────────────────────────────────
def with_retry(fn, retries=3, delay=0.01, exceptions=(Exception,)):
    '''
    Retry fn up to `retries` times on specified exceptions.
    Uses exponential backoff: delay, 2*delay, 4*delay, ...
    Raises the last exception if all retries fail.
    '''
    last_exc = None
    for attempt in range(retries):
        try:
            return fn()   # call the function — no args needed (use closure/lambda)
        except exceptions as e:
            last_exc = e
            if attempt < retries - 1:
                wait = delay * (2 ** attempt)   # 0.01, 0.02, 0.04 seconds
                print(f"  attempt {attempt+1} failed: {e} — retrying in {wait:.3f}s")
                time.sleep(wait)
    raise last_exc           # all retries exhausted

# Simulated flaky service
call_count = 0
def flaky_service():
    global call_count
    call_count += 1
    if call_count < 3:       # fails first 2 times
        raise ConnectionError(f"server unavailable (attempt {call_count})")
    return f"success on attempt {call_count}"

call_count = 0
result = with_retry(flaky_service, retries=4)
print(f"  result: {result}")

# Fails if retries exhausted
call_count = 0
try:
    with_retry(flaky_service, retries=2)
except ConnectionError as e:
    print(f"  gave up after 2 retries: {e}")

# ── EAFP vs LBYL ─────────────────────────────────────────────────────────────
data = {"key": 42}

# LBYL — two lookups (race condition possible in concurrent code)
if "key" in data:
    val = data["key"]
else:
    val = 0
print(f"LBYL: val={val}")

# EAFP — one lookup, Pythonic
try:
    val = data["missing_key"]
except KeyError:
    val = 0
print(f"EAFP: val={val}")

# dict.get() is the idiom for simple defaults
val = data.get("missing_key", 0)
print(f"dict.get(): val={val}")

# ── Defensive input validation ────────────────────────────────────────────────
def compute_average(values):
    if not isinstance(values, (list, tuple)):
        raise TypeError(f"Expected list/tuple, got {type(values).__name__}")
    if len(values) == 0:
        raise ValueError("Cannot compute average of empty sequence")
    if not all(isinstance(v, (int, float)) for v in values):
        raise ValueError("All values must be numeric")
    return sum(values) / len(values)

test_cases = [
    ([1, 2, 3, 4], None),
    ([], "ValueError"),
    ("hello", "TypeError"),
    ([1, "a", 3], "ValueError"),
]
for inp, expected_err in test_cases:
    try:
        avg = compute_average(inp)
        print(f"  avg({inp}) = {avg}")
    except (TypeError, ValueError) as e:
        print(f"  {type(e).__name__}: {e}")

print("Retry / defensive coding demo complete.")


<a id='6'></a>

## 6. Decision Map & Cheat Sheet

---

```
SITUATION                              TOOL
──────────────────────────────────────────────────────────────────
"clean up no matter what"             finally block
"code after success only"             else block
"multiple error types, different handling" multiple except clauses
"catch and re-raise with context"     raise NewError from original_error
"ignore a specific exception"         contextlib.suppress(ExceptionType)
"retry on transient failure"          with_retry() + exponential backoff
"domain-specific error with data"     custom exception class
"validate at input boundary"          raise early with clear message
"resource cleanup guaranteed"         context manager (with statement)
"simple context manager"              @contextlib.contextmanager
"EAFP — just try it"                  try/except KeyError etc.
```

**Anatomy quick reference:**

```python
try:
    result = risky()        # code that might fail
except ValueError as e:     # specific exception
    handle(e)
except (KeyError, IndexError):  # multiple types
    handle_lookup_error()
else:
    use(result)             # only on success
finally:
    cleanup()               # always

# Context manager
with open("file.txt") as f:
    data = f.read()         # f.close() guaranteed

# Custom @contextmanager
@contextlib.contextmanager
def managed(x):
    setup(x)
    try:
        yield x
    finally:
        teardown(x)
```

**Gotchas:**

```
❌  bare `except:` — catches SystemExit, KeyboardInterrupt
❌  except Exception: pass  — silently swallows all errors
❌  LBYL check + access — two lookups, potential race
❌  retry without backoff — hammers the server
✅  except SpecificError: — catch what you expect
✅  raise from e — chains exceptions, preserves context
✅  finally for cleanup — guaranteed execution
✅  @contextmanager — cleaner than __enter__/__exit__ class
```


```
               🛡️ EXCEPTION HANDLING MAP

               try → except → else → finally
               │        │         │       │
               │        │         │       └─ ALWAYS runs (cleanup)
               │        │         └─ runs only on SUCCESS
               │        └─ catches specific exception types
               └─ code that might fail

               EXCEPTION HIERARCHY
               BaseException
               ├─ SystemExit / KeyboardInterrupt / GeneratorExit
               │  └─ DON'T catch with `except Exception`
               └─ Exception
                  ├─ ValueError, TypeError, IndexError, KeyError
                  ├─ AttributeError, NameError, ZeroDivisionError
                  └─ your custom AppError subclasses

               CUSTOM EXCEPTIONS
               class AppError(Exception): pass
               class ValidationError(AppError): ...

               CONTEXT MANAGERS
               ├─ Class: __enter__ / __exit__
               └─ Function: @contextlib.contextmanager + yield

               PATTERNS
               ├─ Retry: loop + except + backoff
               ├─ EAFP: try/except (Pythonic)
               └─ Validate fast: raise at boundary, not deep inside

---
*End of Exception Handling Guide — Sean Edition*
```
